# Lecture 0 · Live demo — why a naive pipeline dies at 64 clients

**LLM Serving Mastery** · runtime: **Google Colab with a CUDA GPU** (T4 was used for the published rehearsal; Colab does not guarantee a GPU type).

Same model, same GPU, two servers, one load generator:

1. **Naive server** — `transformers.pipeline` behind a tiny OpenAI-compatible HTTP server, one request at a time (a lock around `generate`). This is how most first deployments look.
2. **vLLM** — `vllm serve`, the OpenAI-compatible server with continuous batching and PagedAttention.
3. **Load client** — `asyncio` + streaming; fires 1 / 8 / 32 / 64 concurrent clients and measures **TTFT**, **output tokens/s**, **p95** and **failures** (60 s client timeout).

Before running: write down your prediction — *at 64 clients, how many times more tokens/s will vLLM deliver?*

Total run time ≈ 15–20 min, most of it installing vLLM and the naive server grinding through 64 clients.

## 0 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# vLLM goes into its OWN environment: it pins a newer torch (CUDA 13) than Colab's, and installing it
# into Colab's Python breaks the preinstalled torchaudio/transformers — the naive server needs those intact.
!pip -q install uv==0.8.22 httpx==0.28.1 fastapi==0.116.1 uvicorn==0.35.0
!uv venv -q --python 3.12 /content/vllm-env && uv pip install -q --python /content/vllm-env vllm==0.29.0
!/content/vllm-env/bin/vllm --version

In [ ]:
import os, sys, json, time, asyncio, subprocess, statistics, signal, platform, importlib.metadata, math
import httpx

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # small, ungated, fits a T4 many times over
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
PORT_NAIVE, PORT_VLLM = 8001, 8000
CONCURRENCY = [1, 8, 32, 64]
MAX_TOKENS = 128          # output length per request
CLIENT_TIMEOUT = 60.0     # a client gives up after 60 s — like a real load balancer
REPEATS = 3               # report medians/ranges, not one lucky run
PROMPTS = [
    "Explain in a few sentences why the sky is blue.",
    "Give three tips for writing readable Python code.",
    "What is a hash table? Answer briefly.",
    "Summarize the plot of a heist movie you would like to see.",
    "Why do GPUs help deep learning? Keep it short.",
    "Describe a good breakfast for a busy morning.",
    "What is the difference between a process and a thread?",
    "Write a short haiku about a busy kitchen.",
]
if any(p.poll() is None for p in globals().get('OWNED_SERVERS', [])):
    raise RuntimeError('Stop this notebook-owned server before resetting the experiment')
RESULTS, RAW_REQUESTS = {}, []
OWNED_SERVERS = []
print("config ok")

## 1 · The naive server

A deliberately honest "first deployment": `pipeline` loaded once, a `threading.Lock` around generation (the pipeline is not safe to call concurrently, and one GPU stream at a time is what people get anyway), streaming via `TextIteratorStreamer`, OpenAI-style SSE chunks so the **same** client can talk to both servers.

In [ ]:
NAIVE_SERVER = r'''
import asyncio, json, queue, threading, time, torch, uvicorn
from typing import Any, AsyncIterator
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from transformers import pipeline, TextIteratorStreamer, StoppingCriteria, StoppingCriteriaList

MODEL = "%s"
MODEL_REVISION = "%s"
pipe = pipeline("text-generation", model=MODEL, revision=MODEL_REVISION, torch_dtype=torch.float16, device=0)
tok = pipe.tokenizer
lock = threading.Lock()
app = FastAPI()

@app.get("/v1/models")
def models() -> dict[str, list[dict[str, str]]]:
    """Advertise the model identifier used by the owned demo server.

    Returns
    -------
    dict
        OpenAI-style data list with one model ID; not a readiness benchmark.
    """
    return {"data": [{"id": MODEL}]}

class Cancelled(StoppingCriteria):              # stop generating once the client has gone away
    def __init__(self, ev: threading.Event) -> None:
        """Bind the request's cancellation signal.

        Parameters
        ----------
        ev : threading.Event
            Set when this request disconnects or finishes.

        Returns
        -------
        None
            Stores the event without cancelling unrelated requests.
        """
        self.ev = ev
    def __call__(self, *args: Any, **kwargs: Any) -> bool:
        """Check cancellation through the Transformers callback interface.

        Parameters
        ----------
        args : tuple
            Generation callback tensors, unused by this stop rule.
        kwargs : dict
            Additional callback fields, unused by this stop rule.

        Returns
        -------
        bool
            Whether this request's client has gone away.
        """
        return self.ev.is_set()

@app.post("/v1/chat/completions")
async def chat(req: Request) -> StreamingResponse:
    """Create one queued, single-server-stream teaching request.

    Parameters
    ----------
    req : fastapi.Request
        Local OpenAI-style JSON with messages and bounded max_tokens.

    Returns
    -------
    StreamingResponse
        Text-only SSE response; not a complete production API implementation.
    """
    body = await req.json()
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    gone = threading.Event()
    def work() -> None:
        """Generate for this request while holding the single-worker lock.

        Returns
        -------
        None
            Sends decoded text to this request's streamer; no return value.
        """
        with lock:                                   # one order at a time
            if gone.is_set():                        # the guest left while waiting in the queue
                streamer.end(); return
            pipe(body["messages"], max_new_tokens=body.get("max_tokens", 128), do_sample=False,
                 streamer=streamer, stopping_criteria=StoppingCriteriaList([Cancelled(gone)]))
    threading.Thread(target=work, daemon=True).start()
    async def sse() -> AsyncIterator[str]:
        """Poll the owned text queue without blocking the event loop.

        Yields
        ------
        str
            Framed text deltas followed by DONE; transport chunks are not tokens.
        """
        # Poll the streamer's queue without blocking: no thread per waiting guest (a thread pool of
        # a few workers would starve the one stream that is actually cooking).
        try:
            while True:
                try:
                    piece = streamer.text_queue.get_nowait()
                except queue.Empty:
                    if await req.is_disconnected():  # the guest gave up: stop cooking their order
                        return
                    await asyncio.sleep(0.02)
                    continue
                if piece is streamer.stop_signal:
                    break
                if piece:
                    chunk = {"choices": [{"delta": {"content": piece}, "index": 0}]}
                    yield "data: " + json.dumps(chunk) + "\n\n"
            yield "data: [DONE]\n\n"
        finally:
            gone.set()                               # normal end, disconnect or cancellation
    return StreamingResponse(sse(), media_type="text/event-stream")

uvicorn.run(app, host="0.0.0.0", port=%d, log_level="warning")
''' % (MODEL, MODEL_REVISION, PORT_NAIVE)
_ = open("naive_server.py", "w").write(NAIVE_SERVER)

In [ ]:
from typing import Any, Sequence
import socket

def start(cmd: Sequence[str], log: str) -> subprocess.Popen:
    """Start one owned process group with a private log.

    Parameters
    ----------
    cmd : sequence of str
        Explicit executable/arguments; no shell expansion.
    log : str
        Private runtime log path, not a student result artifact.

    Returns
    -------
    subprocess.Popen
        Owned group leader; never use this handle for unrelated processes.
    """
    if any(p.poll() is None for p in OWNED_SERVERS):
        raise RuntimeError('An owned server is still running; stop it before another start')
    for port in (PORT_NAIVE, PORT_VLLM):
        with socket.socket() as probe:
            if probe.connect_ex(('127.0.0.1', port)) == 0:
                raise RuntimeError(f'Port {port} is occupied; do not stop an unowned process')
    with open(log, 'w') as log_file:
        proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)
    OWNED_SERVERS.append(proc)
    return proc

def gpu_used_mib() -> int:
    """Read the first device's total used memory, not allocator memory.

    Returns
    -------
    int
        Device-wide used MiB; this demo requires one selected GPU.
    """
    out = subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
                         shell=True, capture_output=True, text=True).stdout
    return int(out.strip().splitlines()[0])

def stop(proc: subprocess.Popen) -> None:
    """Stop this demo's owned server before loading the next treatment.

    Parameters
    ----------
    proc : subprocess.Popen
        Owned group leader returned by start, not an arbitrary PID.

    Returns
    -------
    None
        Requires the owned group to stop and device memory to recover.
    """
    if proc not in OWNED_SERVERS:
        raise ValueError('Refuse cleanup of an unowned process')
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except ProcessLookupError:
            break
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            continue
    if proc.poll() is None:
        raise RuntimeError('Owned server did not stop; do not load the next model')
    for _ in range(60):                              # the next server needs the VRAM back
        if gpu_used_mib() < 600:
            break
        time.sleep(1)
    used = gpu_used_mib()
    if used >= 600:
        raise RuntimeError(f'Cleanup not verified: GPU still uses {used} MiB; inspect owned logs')
    print(f"owned server stopped; device memory recovered: {used} MiB")

def wait_ready(port: int, proc: subprocess.Popen, log: str, timeout: float = 900) -> None:
    """Wait for the owned server's model endpoint within a wall budget.

    Parameters
    ----------
    port : int
        Loopback TCP port of this treatment.
    proc : subprocess.Popen
        Owned server process, monitored for early exit.
    log : str
        Private diagnostic log shown locally on startup failure.
    timeout : float
        Startup wall budget in seconds, not a request latency threshold.

    Returns
    -------
    None
        Ready endpoint; early exit raises RuntimeError, expiry TimeoutError.
    """
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            print(open(log).read()[-3000:]); raise RuntimeError("server died — log above")
        try:
            if httpx.get(f"http://localhost:{port}/v1/models", timeout=2).status_code == 200:
                print(f"ready on :{port} after {time.time()-t0:.0f} s"); return
        except Exception:
            pass
        time.sleep(3)
    stop(proc)
    raise TimeoutError(f"server on :{port} not ready; owned server stopped")

NAIVE_LAUNCH = [sys.executable, "naive_server.py"]
naive = start(NAIVE_LAUNCH, "naive.log")
wait_ready(PORT_NAIVE, naive, "naive.log")

## 2 · The load client

Every client sends **one** streaming chat request at the same moment. Per request we record:

- **TTFT** — time until the first content chunk arrives; the primary p95 uses successful requests only, matching the judge;
- **tokens** — output tokens (re-tokenized, so both servers are counted the same way);
- **ok / failed** — failed = timeout or error.

Aggregate **successful-output goodput** = total output tokens of successful requests ÷ wall time of the whole wave. Tokens from requests that later fail are excluded. Latency percentiles contain successful requests only; failures are always reported separately.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL, revision=MODEL_REVISION)

async def one_request(client: httpx.AsyncClient, port: int, server: str, repeat: int, concurrency: int, i: int) -> dict[str, Any]:
    """Measure one attempted request with the historical metric-v2 client.

    Parameters
    ----------
    client : httpx.AsyncClient
        Already opened client shared by the concurrent wave.
    port : int
        Treatment's loopback port.
    server : str
        Stable treatment name in raw evidence.
    repeat : int
        Zero-based measured repeat, or -1 for excluded warmup.
    concurrency : int
        Number of attempted clients in this wave.
    i : int
        Client index selecting a literal prompt modulo eight.

    Returns
    -------
    dict
        Attempt identity, ok/error, client first-content/latency seconds and
        retokenized output count. This parser is not a full SSE lifecycle validator.
    """
    body = {"model": MODEL, "max_tokens": MAX_TOKENS, "temperature": 0, "stream": True,
            "messages": [{"role": "user", "content": PROMPTS[i % len(PROMPTS)]}]}
    base = {"request_id": f"{server}-r{repeat}-c{concurrency}-i{i}", "server": server,
            "repeat": repeat, "concurrency": concurrency, "prompt_id": i % len(PROMPTS),
            "started_utc": time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
    t0 = time.perf_counter(); ttft = None; text = []
    try:
        async with asyncio.timeout(CLIENT_TIMEOUT):
            async with client.stream("POST", f"http://localhost:{port}/v1/chat/completions", json=body) as r:
                r.raise_for_status()
                async for line in r.aiter_lines():
                    if not line.startswith("data: ") or line.endswith("[DONE]"):
                        continue
                    delta = json.loads(line[6:])["choices"][0]["delta"].get("content") or ""
                    if delta and ttft is None: ttft = time.perf_counter() - t0
                    text.append(delta)
        joined = "".join(text)
        if ttft is None:
            return {**base, "ok": False, "ttft": None, "tokens": 0, "latency": time.perf_counter() - t0, "err": "EmptyResponse"}
        return {**base, "ok": True, "ttft": ttft, "tokens": len(tok.encode(joined)),
                "latency": time.perf_counter() - t0, "err": None}
    except Exception as e:
        return {**base, "ok": False, "ttft": None, "tokens": 0, "latency": time.perf_counter() - t0, "err": type(e).__name__}

def p95(xs: Sequence[float]) -> float:
    """Compute the historical metric-v2 order-statistic convention.

    Parameters
    ----------
    xs : sequence of float
        Successful-request TTFT values in seconds, not all-attempt latency.

    Returns
    -------
    float
        Sorted round(0.95*(N-1)) entry, or NaN for an empty population.
    """
    xs = sorted(xs); return xs[min(len(xs) - 1, int(round(0.95 * (len(xs) - 1))))] if xs else float("nan")

async def wave(port: int, n: int, server: str, repeat: int, record: bool = True) -> dict[str, Any]:
    """Execute one concurrent wave and retain its all-attempt raw rows.

    Parameters
    ----------
    port : int
        Owned treatment's loopback port.
    n : int
        Number of simultaneous clients.
    server : str
        Treatment identifier.
    repeat : int
        Repeat index; -1 denotes an unrecorded warmup/check.
    record : bool
        Append attempted rows to RAW_REQUESTS when True.

    Returns
    -------
    dict
        Success/failure counts, wave wall seconds, successful-output goodput
        and success-only TTFT percentiles; empty successes yield NaN.
    """
    limits = httpx.Limits(max_connections=n + 8)
    async with httpx.AsyncClient(timeout=None, limits=limits) as client:
        t0 = time.perf_counter()
        res = await asyncio.gather(*[one_request(client, port, server, repeat, n, i) for i in range(n)])
        wall = time.perf_counter() - t0
    ok = [r for r in res if r["ok"]]
    ttfts = [r["ttft"] for r in ok if r["ttft"] is not None]
    if record: RAW_REQUESTS.extend(res)
    return {"clients": n, "ok": len(ok), "failed": n - len(ok), "wall_s": wall,
            "tok_per_s": sum(r["tokens"] for r in ok) / wall,
            "ttft_p50_success": statistics.median(ttfts) if ttfts else float("nan"), "ttft_p95_success": p95(ttfts)}

async def run_all(name: str, port: int, proc: subprocess.Popen) -> None:
    """Run the unchanged concurrency/repeat matrix for one treatment.

    Parameters
    ----------
    name : str
        Treatment name stored in RESULTS and raw request identities.
    port : int
        Owned server's loopback port.
    proc : subprocess.Popen
        Running owned server, checked before warmup.

    Returns
    -------
    None
        Stores raw rows and per-concurrency summaries; warmup is excluded.
    """
    assert proc.poll() is None, f"{name} server is not running — look at its log above before measuring"
    if name in RESULTS or any(r['server'] == name for r in RAW_REQUESTS):
        raise RuntimeError('Prior or partial treatment exists; export it, stop owned servers, then reset config for a fresh attempt')
    RESULTS[name] = {}  # a partial attempt is not a completed measurement
    trials = {n: [] for n in CONCURRENCY}
    await wave(port, 1, name, -1, record=False)  # warm-up, not recorded
    for repeat in range(REPEATS):
        for n in CONCURRENCY:
            m = await wave(port, n, name, repeat); trials[n].append(m)
            print(f"{name:8s} repeat={repeat+1}/{REPEATS} c={n:3d} {m['tok_per_s']:8.1f} tok/s p95 {m['ttft_p95_success']:6.2f} s ok {m['ok']}/{n}")
    RESULTS[name] = {}
    for n, ms in trials.items():
        raw = [r for r in RAW_REQUESTS if r['server'] == name and r['concurrency'] == n and r['ok']]
        ttfts, rates = [r['ttft'] for r in raw if r['ttft'] is not None], [m['tok_per_s'] for m in ms]
        RESULTS[name][n] = {"clients": n, "repeats": REPEATS, "attempted": n * REPEATS,
            "ok": sum(m['ok'] for m in ms), "failed": sum(m['failed'] for m in ms),
            "tok_per_s": statistics.median(rates), "tok_per_s_min": min(rates), "tok_per_s_max": max(rates),
            "wall_s_median": statistics.median(m['wall_s'] for m in ms),
            "ttft_p50_success": statistics.median(ttfts) if ttfts else float('nan'), "ttft_p95_success": p95(ttfts)}


## 3 · Run the naive server

Watch the p95 TTFT column as concurrency grows, and the `ok` column at 64. Every client waits for all the orders in front of it.

In [ ]:
await run_all("pipeline", PORT_NAIVE, naive)
stop(naive)      # free the GPU for vLLM

**Read it before moving on.** Does successful-output goodput grow with the number of clients? It should stay roughly flat: the server does the same amount of work per second no matter how many people are waiting, so extra clients only add waiting time. The p95 TTFT grows roughly linearly with the number of clients. At 64 clients some requests probably hit the 60 s timeout, and those guests simply left.

## 4 · The same model on vLLM

`--dtype half` because the T4 (Turing) has no bfloat16. `--max-model-len 2048` keeps the KV-cache reservation modest.

In [ ]:
VLLM_LAUNCH = ["/content/vllm-env/bin/vllm", "serve", MODEL, "--revision", MODEL_REVISION, "--port", str(PORT_VLLM), "--dtype", "half",
               "--max-model-len", "2048", "--gpu-memory-utilization", "0.85"]
vllm = start(VLLM_LAUNCH, "vllm.log")
wait_ready(PORT_VLLM, vllm, "vllm.log")

After the recorded benchmark waves, we run one dedicated 64-client verification wave while polling vLLM's Prometheus endpoint `/metrics`. We record `vllm:num_requests_running`, the number of requests in the current batch. This extra wave is not included in the table; it is a direct mechanism check.

In [ ]:
import re

async def poll_running(stop_evt: asyncio.Event, samples: list[float]) -> None:
    """Observe the running-request gauge during a separate mechanism wave.

    Parameters
    ----------
    stop_evt : asyncio.Event
        Set by this demo after the mechanism wave completes.
    samples : list[float]
        Mutable list of observed running-request counts, not token counts.

    Returns
    -------
    None
        Appends observations; absent metrics leave the list empty, not zero.
    """
    async with httpx.AsyncClient() as c:
        while not stop_evt.is_set():
            try:
                txt = (await c.get(f"http://localhost:{PORT_VLLM}/metrics", timeout=2)).text
                m = re.search(r'^vllm:num_requests_running\{[^}]*\}\s+([0-9.]+)', txt, re.M)
                if m: samples.append(float(m.group(1)))
            except Exception:
                pass
            await asyncio.sleep(0.25)

await run_all("vLLM", PORT_VLLM, vllm)

samples, evt = [], asyncio.Event()
poller = asyncio.create_task(poll_running(evt, samples))
await wave(PORT_VLLM, 64, "vLLM-mechanism-check", -1, record=False)
evt.set(); await poller
VLLM_MAX_RUNNING = max(samples) if samples else None
print("max requests running in one batch during the 64-client wave:", VLLM_MAX_RUNNING if VLLM_MAX_RUNNING is not None else "n/a")

## 5 · The table for the slide

In [ ]:
nvidia = subprocess.run(['nvidia-smi', '-q'], capture_output=True, text=True).stdout
vllm_python = subprocess.run(['/content/vllm-env/bin/python', '--version'], capture_output=True, text=True).stdout.strip()
public_head = httpx.get('https://api.github.com/repos/levshaazz/llm-serving-mastery-submission-example/commits/main', timeout=15)
public_head.raise_for_status()
public_repository_head_at_export = public_head.json()['sha']
artifact = {"provenance": {"metric_contract": 2, "public_repository_head_at_export": public_repository_head_at_export,
    "finished_utc": time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in ['torch', 'transformers', 'httpx', 'fastapi', 'uvicorn']},
    "model": MODEL, "model_revision": MODEL_REVISION, "vllm": "0.29.0", "vllm_python": vllm_python, "nvidia_smi_q": nvidia,
    "launch_commands": {"pipeline": NAIVE_LAUNCH, "vLLM": VLLM_LAUNCH},
    "workload": {"concurrency": CONCURRENCY, "max_tokens": MAX_TOKENS, "client_timeout_s": CLIENT_TIMEOUT, "repeats": REPEATS},
    "latency_population": "successful requests only; failures reported separately",
    "token_count_method": "retokenized_text_estimate",
    "mechanism_check": {"metric": "vllm:num_requests_running", "wave_clients": 64, "max_observed": VLLM_MAX_RUNNING}},
    "raw_requests": RAW_REQUESTS, "summary": RESULTS}
def json_safe(value: Any) -> Any:
    """Convert nonfinite absent-population markers to JSON null recursively.

    Parameters
    ----------
    value : JSON-like object
        Nested result data; not a source of executable code.

    Returns
    -------
    JSON-like object
        Same nested structure, with nonfinite floats replaced by None.
    """
    if isinstance(value, float) and not math.isfinite(value): return None
    if isinstance(value, dict): return {k: json_safe(v) for k, v in value.items()}
    if isinstance(value, list): return [json_safe(v) for v in value]
    return value
json.dump(json_safe(artifact), open("lecture0_demo_results.json", "w"), indent=2, allow_nan=False)
print(f"wrote {len(RAW_REQUESTS)} raw request records; public repository HEAD at export: {public_repository_head_at_export}")

### Read the matched comparison

One row per concurrency. Goodput excludes tokens from failed requests; TTFT percentiles use successful requests only. Inspect failures alongside speed ratios. A missing ratio is not a zero speedup. Rerunning this viewer does not restart either server.


In [ ]:
from IPython.display import display
import pandas as pd
comparison_rows = []
for n in CONCURRENCY:
    a, b = RESULTS['pipeline'][n], RESULTS['vLLM'][n]
    comparison_rows.append(dict(clients=n, pipeline_tok_s=a['tok_per_s'], vllm_tok_s=b['tok_per_s'],
        ratio=b['tok_per_s']/a['tok_per_s'] if a['tok_per_s'] else None,
        pipeline_p95_s=a['ttft_p95_success'], vllm_p95_s=b['ttft_p95_success'],
        pipeline_failed=a['failed'], vllm_failed=b['failed']))
display(pd.DataFrame(comparison_rows))


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for name, style in [("pipeline", "o--"), ("vLLM", "o-")]:
    ax[0].plot(CONCURRENCY, [RESULTS[name][n]["tok_per_s"] for n in CONCURRENCY], style, label=name)
    ax[1].plot(CONCURRENCY, [RESULTS[name][n]["ttft_p95_success"] for n in CONCURRENCY], style, label=name)
ax[0].set(title="Successful-output goodput", xlabel="concurrent clients", ylabel="completed output tokens / s")
ax[1].set(title="p95 TTFT — successful requests", xlabel="concurrent clients", ylabel="seconds")
for a in ax: a.set_xscale("log", base=2); a.set_xticks(CONCURRENCY, CONCURRENCY); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.show()

**What to read off the plots**

- **Successful-output goodput.** The pipeline's line stays roughly flat. vLLM's line keeps rising with concurrency: one decode step reads the weights once and produces a token for every request in the batch (slide 11). This metric excludes partial output from failed requests, so read it as delivered goodput rather than raw engine throughput.
- **p95 TTFT.** This curve uses successful requests only, so always read it together with the failure count. The pipeline's p95 grows roughly linearly while requests still complete. In this simultaneous-arrival experiment, vLLM admits the wave into continuous batching; it does not by itself demonstrate the latency of staggered arrivals joining an already-running batch.
- **Compare with your prediction.** Did you over- or underestimate the gap? A T4 has about 320 GB/s of peak memory bandwidth and the judge's RTX 5070 Ti about 896 GB/s. That ratio alone does not predict either throughput, latency, or curve shape: kernels, scheduler, CPU, and workload can change the bottleneck. Measure on the target GPU before claiming transfer.

Topics 5–7 explain *why* these curves look this way and how to move them.

In [ ]:
stop(vllm)